In [10]:
# ============================================================
# Section 1: Imports and Test 2 settings
# ============================================================

from pathlib import Path
from datetime import datetime
import importlib

import osxphotos

# Reload helper module without restarting the Jupyter kernel.
#
# This is important because:
# - In Jupyter, `from explorephotoslibrary import *` does NOT automatically
#   pick up edits made to explorephotoslibrary.py after the first import.
# - Restarting the kernel would lose the current notebook state.
# - Reloading the module here lets later cells use the updated functions.
import explorephotoslibrary as _explorephotoslibrary
importlib.reload(_explorephotoslibrary)

from explorephotoslibrary import *

USE_INVENTORY_CACHE = True

# Rebuild only the libraries listed here.
# Usually:
# FORCE_REBUILD_INVENTORY_KEYS = set()
#
# If you changed both backup and current Photos Libraries, use:
# FORCE_REBUILD_INVENTORY_KEYS = {"backup_20250317", "current_default"}
FORCE_REBUILD_INVENTORY_KEYS = {
    "current_default",
    "backup_20250317",
}

# Set to True only when you want to pick Photos Library paths again.
# If False, Test 2 reuses paths saved in data/local_config/test2_library_paths.json.
FORCE_RESELECT_LIBRARY_PATHS = False

TEST2_LIBRARY_PROMPTS = {
    "backup_20250317": "Select BACKUP Photos Library: backup_20250317",
    "current_default": "Select CURRENT default Photos Library: current_default",
}

TEST2_DEFAULT_INITIAL_DIRS = {
    "backup_20250317": "/Volumes",
    "current_default": str(Path.home() / "Pictures"),
}

In [2]:
# ============================================================
# Section 2: Load or build inventories
# ============================================================

TEST2_LIBRARY_HISTORY_PATH = Path("data/local_config/test2_library_paths.json")


def get_test2_library_path(library_key):
    library_history = load_json_file(TEST2_LIBRARY_HISTORY_PATH, default={}) or {}
    saved_library_path = library_history.get(library_key)

    if (
        saved_library_path
        and Path(saved_library_path).exists()
        and not FORCE_RESELECT_LIBRARY_PATHS
    ):
        library_path = Path(saved_library_path)

        print("=" * 80)
        print(f"Use saved Photos Library path for: {library_key}")
        print("=" * 80)
        print(f"{library_key} library path:", library_path)
        print()

        return library_path

    if saved_library_path:
        initial_dir = Path(saved_library_path).parent
    else:
        initial_dir = Path(TEST2_DEFAULT_INITIAL_DIRS.get(library_key, "/Volumes"))

    prompt = TEST2_LIBRARY_PROMPTS.get(
        library_key,
        f"Select Photos Library for: {library_key}",
    )

    print("=" * 80)
    print(prompt)
    print("=" * 80)

    library_path = Path(
        choose_photos_library_path(
            initial_dir=initial_dir,
            prompt=prompt,
        )
    )

    library_history[library_key] = str(library_path)
    library_history[f"{library_key}_selected_at"] = datetime.now().isoformat()
    save_json_file(TEST2_LIBRARY_HISTORY_PATH, library_history)

    print(f"{library_key} library path:", library_path)
    print()

    return library_path


def load_or_build_inventory(library_key):
    library_path = get_test2_library_path(library_key)
    should_rebuild_inventory = library_key in FORCE_REBUILD_INVENTORY_KEYS

    if USE_INVENTORY_CACHE and not should_rebuild_inventory:
        print("=" * 80)
        print(f"Load inventory cache: {library_key}")
        print("=" * 80)

        try:
            return load_inventory_cache(library_key)
        except FileNotFoundError:
            print(f"Cache not found for {library_key}. Build inventory instead.")
            print()

    if should_rebuild_inventory:
        print("=" * 80)
        print(f"Force rebuild inventory: {library_key}")
        print("=" * 80)
    else:
        print("=" * 80)
        print(f"Build inventory: {library_key}")
        print("=" * 80)

    osx_assets = osxphotos.PhotosDB(str(library_path)).photos()
    print(f"{library_key} osx asset count:", len(osx_assets))

    inventory = build_inventory(osx_assets)

    print()
    print(f"{library_key} inventory summary")
    print("-" * 80)
    print_inventory_summary(inventory)

    save_inventory_cache(inventory, library_key)

    return inventory


inventory_backup = load_or_build_inventory("backup_20250317")

print()

inventory_current = load_or_build_inventory("current_default")

Use saved Photos Library path for: backup_20250317
backup_20250317 library path: /Volumes/PRO-G40-0605/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary

Force rebuild inventory: backup_20250317
backup_20250317 osx asset count: 71572
processed assets: 10000
processed assets: 20000
processed assets: 30000
processed assets: 40000
processed assets: 50000
processed assets: 60000
processed assets: 70000

backup_20250317 inventory summary
--------------------------------------------------------------------------------
inventory assets: 71572
inventory albums: 5171
inventory folders: 35
special assets:
  PATH_MISSING: 0
  PATH_NONE: 0
  SYNDICATED_NO_NORMAL_ORIGINAL: 0
  UNKNOWN_PATH: 0
movies: 6224
hidden: 0
favorites: 702
descriptions: 728
keywords: 23747
saved inventory cache: data/inventory_cache/backup_20250317.inventory.pkl.gz
elapsed seco

In [3]:
# ============================================================
# Section 3: Photo Library asset unique ID audit
# ============================================================

print("=" * 120)
print("Section 3: Photo Library asset unique ID audit")
print("=" * 120)

print()
backup_unique_ok = audit_photo_library_asset_unique_ids(
    inventory_backup,
    label="BACKUP",
)

print()
current_unique_ok = audit_photo_library_asset_unique_ids(
    inventory_current,
    label="CURRENT",
)

if not backup_unique_ok:
    raise RuntimeError("BACKUP photo_library_asset_unique_id audit failed.")

if not current_unique_ok:
    raise RuntimeError("CURRENT photo_library_asset_unique_id audit failed.")

print()
print("Photo Library asset unique ID audit passed.")

Section 3: Photo Library asset unique ID audit

BACKUP
--------------------------------------------------------------------------------
total asset count: 71572
generated unique ID count: 71572
assets without unique ID: 0
duplicate unique ID group count: 0
duplicate asset count: 0
is Photo Library asset unique ID scheme unique: True

CURRENT
--------------------------------------------------------------------------------
total asset count: 95333
generated unique ID count: 95333
assets without unique ID: 0
duplicate unique ID group count: 0
duplicate asset count: 0
is Photo Library asset unique ID scheme unique: True

Photo Library asset unique ID audit passed.


In [11]:
# ============================================================
# Section 4: Run inventory comparison summary
# ============================================================

if not (backup_unique_ok and current_unique_ok):
    raise RuntimeError(
        "Photo Library asset unique ID audit failed. "
        "Do not run cross-library comparison yet."
    )

diff_records = compare_inventories(
    inventory_backup=inventory_backup,
    inventory_current=inventory_current,
)

summarize_diff_records(diff_records)

diff record count: 59337

change_type counts
--------------------------------------------------------------------------------
ALBUM_FOLDER_PATHS_CHANGED: 318
ALBUM_MISSING_FROM_CURRENT: 47
ALBUM_NEW_IN_CURRENT: 853
ASSET_ALBUM_MEMBERSHIP_CHANGED: 11881
ASSET_FOLDER_PATHS_CHANGED: 18504
ASSET_METADATA_CHANGED: 3965
ASSET_NEW_IN_CURRENT: 23761
FOLDER_MISSING_FROM_CURRENT: 5
FOLDER_NEW_IN_CURRENT: 3

scope counts
--------------------------------------------------------------------------------
album: 1218
asset: 58111
folder: 8


In [12]:
# ============================================================
# Section 5: Print ASSET_MISSING_FROM_CURRENT review list
# ============================================================

missing_current_review_result = print_missing_current_review(
    diff_records=diff_records,
    inventory_current=inventory_current,
    max_current_candidates=5,
)

# Optional: write text/TSV files only when you explicitly want files.
WRITE_MISSING_CURRENT_REVIEW_REPORT_FILES = False

if WRITE_MISSING_CURRENT_REVIEW_REPORT_FILES:
    missing_current_review_file_result = write_missing_current_review_report(
        diff_records=diff_records,
        inventory_current=inventory_current,
        output_dir=Path("reports/test2_missing_current_review"),
        report_name_prefix="asset_missing_from_current_review",
    )
    print_missing_current_review_report_summary(missing_current_review_file_result)


Section 5: Compact ASSET_MISSING_FROM_CURRENT review list
matched record count: 0

Quick counts
------------------------------------------------------------------------------------------------------------------------
is_movie: {}
hasadjustments: {}
favorite: {}
hidden: {}
path_exists: {}

One-by-one manual verification checklist



In [17]:
# ============================================================
# Section 6: Export one album repair manifest for PhotoKit
# ============================================================

from pathlib import Path
from datetime import datetime
import csv
import re
import shutil


# ------------------------------------------------------------
# User target
# ------------------------------------------------------------

TARGET_FOLDER_PATH = "NSFW"
TARGET_ALBUM_TITLE = "#NSFW #Line色色群組 #懶得分類亂七八糟放在一起 2024年5月21日（2024年10月28日 繼續加內容）"

# For nested folders, use either:
#   TARGET_FOLDER_PATH = "NSFW / HIDE"
# or:
#   TARGET_FOLDER_PATH = "NSFW/HIDE"
#
# The code normalizes both forms to:
#   NSFW / HIDE


# ------------------------------------------------------------
# Output protocol
# ------------------------------------------------------------

REPAIR_INBOX_DIR = Path.home() / "Downloads" / "PhotosRepairMVP_Inbox"
LATEST_MANIFEST_PATH = REPAIR_INBOX_DIR / "RepairManifestLatest.tsv"

WRITE_ARCHIVE_COPY = True
ARCHIVE_DIR = REPAIR_INBOX_DIR / "archive"

MAX_PREVIEW_ROWS = 80


# ------------------------------------------------------------
# Path helpers
# ------------------------------------------------------------

def _normalize_photo_folder_path(path):
    if path is None:
        return ""

    text = str(path).strip()
    text = text.replace("\\", "/")
    text = text.replace(" / ", "/")

    parts = [
        part.strip()
        for part in text.split("/")
        if part.strip()
    ]

    return " / ".join(parts)


def _join_photo_path(folder_path, album_title):
    folder = _normalize_photo_folder_path(folder_path)
    album = str(album_title).strip()

    if folder:
        return f"{folder} / {album}"

    return album


def _safe_filename_part(text):
    text = str(text).strip()
    text = text.replace(" / ", "__")
    text = text.replace("/", "__")
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_") or "untitled"


NORMALIZED_TARGET_FOLDER_PATH = _normalize_photo_folder_path(TARGET_FOLDER_PATH)
TARGET_ALBUM_PATH = _join_photo_path(
    NORMALIZED_TARGET_FOLDER_PATH,
    TARGET_ALBUM_TITLE,
)


# ------------------------------------------------------------
# Inventory helpers
# ------------------------------------------------------------

def _deepest_folder_path_for_album(album):
    folders = album.get("folders") or {}

    folder_paths = [
        _normalize_photo_folder_path(folder.get("path") or folder.get("title"))
        for folder in folders.values()
        if folder.get("path") or folder.get("title")
    ]

    folder_paths = [
        path
        for path in folder_paths
        if path
    ]

    if not folder_paths:
        return None

    return sorted(
        folder_paths,
        key=lambda value: (value.count("/"), len(value)),
    )[-1]


def _full_album_path(album):
    album_title = album.get("title")

    if not album_title:
        return None

    folder_path = _deepest_folder_path_for_album(album)

    if folder_path:
        return _join_photo_path(folder_path, album_title)

    return str(album_title).strip()


def _asset_album_paths_for_section6(asset):
    album_paths = []

    for album in (asset.get("albums") or {}).values():
        album_path = _full_album_path(album)

        if album_path:
            album_paths.append(album_path)

    return tuple(sorted(set(album_paths)))


def _asset_in_album_path(asset, album_path):
    return album_path in _asset_album_paths_for_section6(asset)


def _asset_unique_id_key(asset):
    unique_id = asset.get("photo_library_asset_unique_id")

    if unique_id is None:
        return None

    return tuple(unique_id)


def _format_dt(value):
    if value is None:
        return "-"

    if hasattr(value, "strftime"):
        return value.strftime("%Y-%m-%d %H:%M:%S")

    text = str(value)
    text = text.replace("T", " ")

    if "+" in text:
        text = text.split("+", 1)[0]

    return text[:19]


def _asset_media_label(asset):
    return "video" if asset.get("is_movie") else "photo"


def _asset_original_size(asset):
    width = asset.get("original_width")
    height = asset.get("original_height")

    if width is None or height is None:
        return "-"

    return f"{width}x{height}"


def _tsv_clean(value):
    if value is None:
        return "-"

    text = str(value)
    text = text.replace("\t", " ")
    text = text.replace("\r", " ")
    text = text.replace("\n", " ")
    return text


def _build_current_asset_index_by_unique_id(inventory_current):
    index = {}

    for asset in inventory_current.get("assets") or []:
        key = _asset_unique_id_key(asset)

        if key is None:
            continue

        if key in index:
            raise RuntimeError(f"Duplicate current unique_id: {key}")

        index[key] = asset

    return index


def _print_album_paths_with_same_title(inventory, label, album_title):
    print(f"{label} album paths with title == {album_title!r}")
    print("-" * 80)

    paths = sorted(
        path
        for album in (inventory.get("albums") or {}).values()
        for path in [_full_album_path(album)]
        if album.get("title") == album_title and path
    )

    if not paths:
        print("  -")
    else:
        for path in paths:
            print(f"  {path}")

    print()


# ------------------------------------------------------------
# Build repair rows
# ------------------------------------------------------------

print("=" * 120)
print("Section 6: Export one album repair manifest for PhotoKit")
print("=" * 120)
print()
print("Target backup album path:")
print(f"  {TARGET_ALBUM_PATH}")
print()
print("Output:")
print(f"  latest:  {LATEST_MANIFEST_PATH}")
print(f"  archive: {ARCHIVE_DIR if WRITE_ARCHIVE_COPY else '(disabled)'}")
print()

_print_album_paths_with_same_title(
    inventory=inventory_backup,
    label="BACKUP",
    album_title=TARGET_ALBUM_TITLE,
)

_print_album_paths_with_same_title(
    inventory=inventory_current,
    label="CURRENT",
    album_title=TARGET_ALBUM_TITLE,
)

backup_assets_in_album = [
    asset
    for asset in inventory_backup.get("assets") or []
    if _asset_in_album_path(asset, TARGET_ALBUM_PATH)
]

backup_assets_in_album.sort(
    key=lambda asset: (
        _format_dt(asset.get("date")),
        asset.get("original_filename") or "",
        asset.get("uuid") or "",
    )
)

current_by_unique_id = _build_current_asset_index_by_unique_id(inventory_current)

repair_rows = []

for backup_asset in backup_assets_in_album:
    key = _asset_unique_id_key(backup_asset)
    current_asset = current_by_unique_id.get(key) if key is not None else None

    repair_rows.append({
        "operation": "add_asset_to_album",
        "folder_path": NORMALIZED_TARGET_FOLDER_PATH,
        "album_title": TARGET_ALBUM_TITLE,
        "media": _asset_media_label(backup_asset),
        "original_filename": backup_asset.get("original_filename") or "-",
        "date": _format_dt(backup_asset.get("date")),
        "original_size": _asset_original_size(backup_asset),
        "backup_uuid": backup_asset.get("uuid") or "-",
        "current_uuid": (current_asset.get("uuid") if current_asset else "-"),
        "status": "MATCHED_IN_CURRENT" if current_asset else "MISSING_IN_CURRENT",
    })


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("=" * 120)
print("Backup album asset summary")
print("=" * 120)

photo_count = sum(row["media"] == "photo" for row in repair_rows)
video_count = sum(row["media"] == "video" for row in repair_rows)
matched_count = sum(row["status"] == "MATCHED_IN_CURRENT" for row in repair_rows)
missing_count = sum(row["status"] == "MISSING_IN_CURRENT" for row in repair_rows)

print(f"backup album path: {TARGET_ALBUM_PATH}")
print(f"backup asset count: {len(repair_rows)}")
print(f"photos: {photo_count}")
print(f"videos: {video_count}")
print(f"matched in current: {matched_count}")
print(f"missing in current: {missing_count}")
print()

print("=" * 120)
print(f"Preview rows, first {MAX_PREVIEW_ROWS}")
print("=" * 120)

for index, row in enumerate(repair_rows[:MAX_PREVIEW_ROWS], start=1):
    print(
        f"{index:04d}. "
        f"{row['status']} | "
        f"{row['media']} | "
        f"{row['original_filename']} | "
        f"{row['date']} | "
        f"{row['original_size']} | "
        f"backup={row['backup_uuid']} | "
        f"current={row['current_uuid']}"
    )

if len(repair_rows) > MAX_PREVIEW_ROWS:
    print()
    print(f"... {len(repair_rows) - MAX_PREVIEW_ROWS} more rows not printed")
    print()


# ------------------------------------------------------------
# Write TSV manifest
# ------------------------------------------------------------

header = [
    "operation",
    "folder_path",
    "album_title",
    "media",
    "original_filename",
    "date",
    "original_size",
    "backup_uuid",
    "current_uuid",
    "status",
]

REPAIR_INBOX_DIR.mkdir(parents=True, exist_ok=True)

with LATEST_MANIFEST_PATH.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=header,
        delimiter="\t",
        lineterminator="\n",
        extrasaction="ignore",
    )

    writer.writeheader()

    for row in repair_rows:
        writer.writerow({
            key: _tsv_clean(row.get(key))
            for key in header
        })

archive_path = None

if WRITE_ARCHIVE_COPY:
    ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    archive_filename = (
        f"{timestamp}__"
        f"{_safe_filename_part(NORMALIZED_TARGET_FOLDER_PATH)}__"
        f"{_safe_filename_part(TARGET_ALBUM_TITLE)}.tsv"
    )

    archive_path = ARCHIVE_DIR / archive_filename
    shutil.copy2(LATEST_MANIFEST_PATH, archive_path)


print("=" * 120)
print("Manifest written")
print("=" * 120)
print(f"latest path: {LATEST_MANIFEST_PATH}")

if archive_path:
    print(f"archive path: {archive_path}")

print()
print("Manifest protocol:")
print("\t".join(header))
print()
print("Ready for PhotosRepairMVP macOS app.")

Section 6: Export one album repair manifest for PhotoKit

Target backup album path:
  NSFW / #NSFW #Line色色群組 #懶得分類亂七八糟放在一起 2024年5月21日（2024年10月28日 繼續加內容）

Output:
  latest:  /Users/huohsien/Downloads/PhotosRepairMVP_Inbox/RepairManifestLatest.tsv
  archive: /Users/huohsien/Downloads/PhotosRepairMVP_Inbox/archive

BACKUP album paths with title == '#NSFW #Line色色群組 #懶得分類亂七八糟放在一起 2024年5月21日（2024年10月28日 繼續加內容）'
--------------------------------------------------------------------------------
  NSFW / #NSFW #Line色色群組 #懶得分類亂七八糟放在一起 2024年5月21日（2024年10月28日 繼續加內容）

CURRENT album paths with title == '#NSFW #Line色色群組 #懶得分類亂七八糟放在一起 2024年5月21日（2024年10月28日 繼續加內容）'
--------------------------------------------------------------------------------
  -

Backup album asset summary
backup album path: NSFW / #NSFW #Line色色群組 #懶得分類亂七八糟放在一起 2024年5月21日（2024年10月28日 繼續加內容）
backup asset count: 136
photos: 24
videos: 112
matched in current: 136
missing in current: 0

Preview rows, first 80
0001. MATCHED_IN_CURRENT | v

In [27]:
# ============================================================
# Section 6A: List unfinished album-membership repairs under one backup folder
# ============================================================

TARGET_PARENT_FOLDER_PATH = "NSFW"


def s6a_normalize_folder_path(path):
    if path is None:
        return ""

    text = str(path).strip()
    text = text.replace("\\", "/")
    text = text.replace(" / ", "/")

    parts = []
    for part in text.split("/"):
        part = part.strip()
        if part:
            parts.append(part)

    return " / ".join(parts)


def s6a_normalize_filename_extension(filename):
    if filename is None:
        return None

    text = str(filename)

    if "." not in text:
        return text

    stem, extension = text.rsplit(".", 1)

    if not stem or not extension:
        return text

    return stem + "." + extension.lower()


def s6a_normalize_unique_id(unique_id):
    if unique_id is None:
        return None

    if len(unique_id) < 5:
        return tuple(unique_id)

    return (
        s6a_normalize_filename_extension(unique_id[0]),
        unique_id[1],
        unique_id[2],
        unique_id[3],
        unique_id[4],
    )


def s6a_album_folder_path(album):
    folders = album.get("folders") or {}
    paths = []

    for folder in folders.values():
        path = folder.get("path") or folder.get("title")
        path = s6a_normalize_folder_path(path)

        if path:
            paths.append(path)

    if not paths:
        return ""

    return sorted(paths, key=lambda value: (value.count(" / "), len(value)))[-1]


def s6a_album_path(album):
    title = album.get("title")

    if not title:
        return None

    folder_path = s6a_album_folder_path(album)

    if folder_path:
        return folder_path + " / " + str(title)

    return str(title)


def s6a_asset_album_paths(asset):
    result = []

    for album in (asset.get("albums") or {}).values():
        path = s6a_album_path(album)

        if path:
            result.append(path)

    return tuple(sorted(set(result)))


def s6a_asset_key(asset):
    unique_id = asset.get("photo_library_asset_unique_id")

    if unique_id is None:
        return None

    return s6a_normalize_unique_id(unique_id)


def s6a_is_under_target(album_path, target_parent):
    return album_path == target_parent or album_path.startswith(target_parent + " / ")


def s6a_build_membership_map(inventory, target_parent):
    membership = {}

    for asset in inventory.get("assets") or []:
        asset_key = s6a_asset_key(asset)

        if asset_key is None:
            continue

        for album_path in s6a_asset_album_paths(asset):
            if s6a_is_under_target(album_path, target_parent):
                membership.setdefault(album_path, set()).add(asset_key)

    return membership


def s6a_build_backup_media_counts(inventory, target_parent):
    counts = {}

    for asset in inventory.get("assets") or []:
        for album_path in s6a_asset_album_paths(asset):
            if not s6a_is_under_target(album_path, target_parent):
                continue

            row = counts.setdefault(album_path, {"photos": 0, "videos": 0})

            if asset.get("is_movie"):
                row["videos"] += 1
            else:
                row["photos"] += 1

    return counts


target_parent = s6a_normalize_folder_path(TARGET_PARENT_FOLDER_PATH)

backup_membership = s6a_build_membership_map(inventory_backup, target_parent)
current_membership = s6a_build_membership_map(inventory_current, target_parent)
backup_media_counts = s6a_build_backup_media_counts(inventory_backup, target_parent)

unfinished_rows = []
ok_count = 0

for album_path in sorted(backup_membership.keys()):
    backup_members = backup_membership.get(album_path, set())
    current_members = current_membership.get(album_path, set())

    matched = backup_members & current_members
    need_add = backup_members - current_members
    extra = current_members - backup_members

    if len(need_add) == 0 and len(extra) == 0:
        ok_count += 1
        continue

    media_counts = backup_media_counts.get(album_path, {"photos": 0, "videos": 0})

    if len(need_add) > 0 and len(extra) > 0:
        action = "ADD+CHECK"
    elif len(need_add) > 0:
        action = "ADD"
    else:
        action = "CHECK_EXTRA"

    unfinished_rows.append({
        "album_path": album_path,
        "backup": len(backup_members),
        "photos": media_counts["photos"],
        "videos": media_counts["videos"],
        "current": len(current_members),
        "matched": len(matched),
        "need_add": len(need_add),
        "extra": len(extra),
        "action": action,
    })

unfinished_rows.sort(
    key=lambda row: (
        row["action"] != "ADD",
        -row["need_add"],
        -row["backup"],
        row["album_path"],
    )
)

print("=" * 150)
print("Section 6A: Unfinished album-membership repairs under backup folder: {}".format(target_parent))
print("=" * 150)
print()
print("NOTE: This is based on inventory_current cache.")
print("      Rebuild current inventory before using this as final verification.")
print()
print("backup albums total: {}".format(len(backup_membership)))
print("already OK in current cache: {}".format(ok_count))
print("unfinished in current cache: {}".format(len(unfinished_rows)))
print()

print(
    "{:>3}  {:>7}  {:>7}  {:>7}  {:>7}  {:>7}  {:>8}  {:>6}  {:<10}  {}".format(
        "idx",
        "backup",
        "photos",
        "videos",
        "current",
        "matched",
        "need_add",
        "extra",
        "action",
        "album_path",
    )
)

print(
    "{:>3}  {:>7}  {:>7}  {:>7}  {:>7}  {:>7}  {:>8}  {:>6}  {:<10}  {}".format(
        "---",
        "-------",
        "-------",
        "-------",
        "-------",
        "-------",
        "--------",
        "------",
        "----------",
        "-" * 80,
    )
)

for index, row in enumerate(unfinished_rows, start=1):
    print(
        "{:>3}  {:>7}  {:>7}  {:>7}  {:>7}  {:>7}  {:>8}  {:>6}  {:<10}  {}".format(
            index,
            row["backup"],
            row["photos"],
            row["videos"],
            row["current"],
            row["matched"],
            row["need_add"],
            row["extra"],
            row["action"],
            row["album_path"],
        )
    )

Section 6A: Unfinished album-membership repairs under backup folder: NSFW

NOTE: This is based on inventory_current cache.
      Rebuild current inventory before using this as final verification.

backup albums total: 65
already OK in current cache: 3
unfinished in current cache: 62

idx   backup   photos   videos  current  matched  need_add   extra  action      album_path
---  -------  -------  -------  -------  -------  --------  ------  ----------  --------------------------------------------------------------------------------
  1     1594     1586        8        0        0      1594       0  ADD         NSFW / IG Pretty Girls
  2     1074     1025       49        0        0      1074       0  ADD         NSFW / Sexy Girls
  3      960        2      958        0        0       960       0  ADD         NSFW / Chaturbate
  4      897      858       39        1        1       896       0  ADD         NSFW / （隱藏內容）TG Girls -1
  5      387      387        0        0        0       387 

In [7]:
# # ============================================================
# # Appendix A: Duplicate diagnostic archive
# # 
# # TEMP: Diagnose potential duplicate groups by SHA256
# #       with full manual-review metadata
# # ============================================================

# import hashlib
# import os
# import time
# from datetime import datetime


# def compute_sha256_for_asset(asset, chunk_size=1024 * 1024):
#     cached_sha256 = asset.get("content_sha256")
#     if cached_sha256:
#         return cached_sha256

#     path = asset.get("path")

#     if path is None:
#         return None

#     if not os.path.exists(path):
#         return None

#     sha256 = hashlib.sha256()

#     with open(path, "rb") as f:
#         while True:
#             chunk = f.read(chunk_size)

#             if not chunk:
#                 break

#             sha256.update(chunk)

#     digest = sha256.hexdigest()
#     asset["content_sha256"] = digest
#     return digest


# def build_potential_duplicate_groups_by_unique_id(inventory):
#     unique_id_to_assets = {}

#     for asset in inventory["assets"]:
#         unique_id = asset.get("photo_library_asset_unique_id")

#         if unique_id is None:
#             continue

#         if unique_id not in unique_id_to_assets:
#             unique_id_to_assets[unique_id] = []

#         unique_id_to_assets[unique_id].append(asset)

#     return {
#         unique_id: assets
#         for unique_id, assets in unique_id_to_assets.items()
#         if len(assets) > 1
#     }


# def normalize_string_list(values):
#     result = []

#     if values is None:
#         return result

#     if isinstance(values, str):
#         return [values]

#     if isinstance(values, dict):
#         iterable = values.values()
#     elif isinstance(values, (list, tuple, set)):
#         iterable = values
#     else:
#         return [str(values)]

#     for item in iterable:
#         if item is None:
#             continue

#         if isinstance(item, str):
#             value = item
#         elif isinstance(item, dict):
#             value = (
#                 item.get("title")
#                 or item.get("name")
#                 or item.get("path")
#                 or item.get("folder_path")
#                 or item.get("album_path")
#             )
#         else:
#             value = str(item)

#         if value:
#             result.append(value)

#     return sorted(set(result))


# def get_asset_album_titles(asset):
#     albums = asset.get("albums")
#     return normalize_string_list(albums)


# def get_asset_folder_paths(asset):
#     folders = asset.get("folders")
#     folder_paths = normalize_string_list(folders)

#     # Some inventory formats may store folder paths under different keys.
#     extra_candidates = [
#         asset.get("folder_paths"),
#         asset.get("folder_path"),
#         asset.get("album_folder_paths"),
#     ]

#     for candidate in extra_candidates:
#         folder_paths.extend(normalize_string_list(candidate))

#     return sorted(set(folder_paths))


# def get_asset_keywords(asset):
#     keywords = asset.get("keywords")
#     return normalize_string_list(keywords)


# def get_asset_description(asset):
#     return (
#         asset.get("description")
#         or asset.get("caption")
#         or asset.get("title")
#         or ""
#     )


# def parse_date_added_for_sort(asset):
#     date_added = asset.get("date_added")

#     if not date_added:
#         return datetime.max

#     if isinstance(date_added, datetime):
#         return date_added

#     text = str(date_added)

#     try:
#         return datetime.fromisoformat(text.replace("Z", "+00:00"))
#     except Exception:
#         return datetime.max


# def asset_metadata_signature(asset):
#     return {
#         "albums": tuple(get_asset_album_titles(asset)),
#         "folders": tuple(get_asset_folder_paths(asset)),
#         "keywords": tuple(get_asset_keywords(asset)),
#         "description": get_asset_description(asset),
#         "favorite": asset.get("favorite"),
#         "hidden": asset.get("hidden"),
#         "hasadjustments": asset.get("hasadjustments"),
#         "adjustment_signature": asset.get("adjustment_signature"),
#     }


# def metadata_score(asset):
#     return (
#         len(get_asset_album_titles(asset)) * 10
#         + len(get_asset_folder_paths(asset)) * 10
#         + len(get_asset_keywords(asset)) * 5
#         + (1 if get_asset_description(asset) else 0)
#         + (1 if asset.get("favorite") else 0)
#         + (1 if asset.get("hidden") else 0)
#     )


# def choose_representative_asset(assets):
#     # Prefer metadata-rich assets; tie-break by earliest Date Added.
#     return sorted(
#         assets,
#         key=lambda asset: (
#             -metadata_score(asset),
#             parse_date_added_for_sort(asset),
#             asset.get("uuid") or "",
#         ),
#     )[0]


# def print_asset_manual_review_block(asset, indent="  "):
#     print(f"{indent}UUID:", asset.get("uuid"))
#     print(f"{indent}Original File Name:", asset.get("original_filename"))
#     print(f"{indent}Filename:", asset.get("filename"))
#     print(f"{indent}Date:", asset.get("date"))
#     print(f"{indent}Date Added:", asset.get("date_added"))
#     print(f"{indent}File Size:", asset.get("file_size_bytes"))
#     print(f"{indent}Has Adjustments:", asset.get("hasadjustments"))
#     print(f"{indent}Adjustment Signature:", asset.get("adjustment_signature"))
#     print(f"{indent}Width x Height:", asset.get("width"), "x", asset.get("height"))
#     print(f"{indent}Original Width x Height:", asset.get("original_width"), "x", asset.get("original_height"))
#     print(f"{indent}Albums:", get_asset_album_titles(asset))
#     print(f"{indent}Folder Paths:", get_asset_folder_paths(asset))
#     print(f"{indent}Keywords:", get_asset_keywords(asset))
#     print(f"{indent}Description:", get_asset_description(asset))
#     print(f"{indent}Favorite:", asset.get("favorite"))
#     print(f"{indent}Hidden:", asset.get("hidden"))
#     print(f"{indent}Path:", asset.get("path"))


# def print_cleanup_recommendation(assets):
#     metadata_signatures = [asset_metadata_signature(asset) for asset in assets]
#     metadata_all_same = all(
#         signature == metadata_signatures[0]
#         for signature in metadata_signatures
#     )

#     representative = choose_representative_asset(assets)

#     if metadata_all_same:
#         print("Recommendation:")
#         print("  Metadata appears identical.")
#         print("  Keep earliest / representative asset:")
#         print("   ", representative.get("uuid"))
#         print("  Delete other duplicate asset(s):")
#         for asset in assets:
#             if asset is not representative:
#                 print("   ", asset.get("uuid"))
#     else:
#         print("Recommendation:")
#         print("  Metadata differs across duplicate assets.")
#         print("  Do NOT blindly delete.")
#         print("  Suggested representative, based on richer metadata + earliest Date Added:")
#         print("   ", representative.get("uuid"))
#         print("  Before deleting others, manually confirm whether album/folder/keyword membership should be preserved.")


# def diagnose_potential_duplicate_groups_with_sha256_and_metadata(
#     inventory,
#     label,
#     max_true_duplicate_groups_to_print=50,
#     max_key_collision_groups_to_print=20,
# ):
#     start_time = time.perf_counter()

#     potential_groups = build_potential_duplicate_groups_by_unique_id(inventory)

#     true_content_duplicate_groups = []
#     key_collision_groups = []
#     sha_error_assets = []

#     checked_asset_count = 0

#     for unique_id, assets in potential_groups.items():
#         sha256_to_assets = {}

#         for asset in assets:
#             checked_asset_count += 1
#             sha256 = compute_sha256_for_asset(asset)

#             if sha256 is None:
#                 sha_error_assets.append(asset)
#                 continue

#             if sha256 not in sha256_to_assets:
#                 sha256_to_assets[sha256] = []

#             sha256_to_assets[sha256].append(asset)

#         duplicate_sha_groups = {
#             sha256: sha_assets
#             for sha256, sha_assets in sha256_to_assets.items()
#             if len(sha_assets) > 1
#         }

#         if duplicate_sha_groups:
#             for sha256, sha_assets in duplicate_sha_groups.items():
#                 true_content_duplicate_groups.append(
#                     {
#                         "unique_id": unique_id,
#                         "sha256": sha256,
#                         "assets": sha_assets,
#                     }
#                 )

#         if len(sha256_to_assets) > 1:
#             key_collision_groups.append(
#                 {
#                     "unique_id": unique_id,
#                     "sha256_to_assets": sha256_to_assets,
#                 }
#             )

#     elapsed = time.perf_counter() - start_time

#     print(label)
#     print("-" * 120)
#     print("potential duplicate unique_id group count:", len(potential_groups))
#     print("checked asset count:", checked_asset_count)
#     print("sha error asset count:", len(sha_error_assets))
#     print("true content duplicate group count:", len(true_content_duplicate_groups))
#     print("key collision group count:", len(key_collision_groups))
#     print("elapsed seconds:", round(elapsed, 3))

#     print()
#     print("TRUE CONTENT DUPLICATE GROUPS — MANUAL REVIEW")
#     print("-" * 120)

#     for index, group in enumerate(true_content_duplicate_groups, start=1):
#         if index > max_true_duplicate_groups_to_print:
#             print("... more true content duplicate groups not printed")
#             break

#         assets_sorted = sorted(
#             group["assets"],
#             key=lambda asset: (
#                 parse_date_added_for_sort(asset),
#                 asset.get("uuid") or "",
#             ),
#         )

#         print("=" * 120)
#         print(f"Group {index:02d}")
#         print("=" * 120)
#         print("unique_id:", group["unique_id"])
#         print("sha256:", group["sha256"])
#         print("asset count:", len(assets_sorted))

#         first_asset = assets_sorted[0]
#         print("Original File Name:", first_asset.get("original_filename"))
#         print("Date:", first_asset.get("date"))
#         print("File Size:", first_asset.get("file_size_bytes"))
#         print("Adjustment Signature:", first_asset.get("adjustment_signature"))

#         union_albums = sorted(
#             set(
#                 album
#                 for asset in assets_sorted
#                 for album in get_asset_album_titles(asset)
#             )
#         )
#         union_folders = sorted(
#             set(
#                 folder
#                 for asset in assets_sorted
#                 for folder in get_asset_folder_paths(asset)
#             )
#         )
#         union_keywords = sorted(
#             set(
#                 keyword
#                 for asset in assets_sorted
#                 for keyword in get_asset_keywords(asset)
#             )
#         )

#         print("Union Albums:", union_albums)
#         print("Union Folder Paths:", union_folders)
#         print("Union Keywords:", union_keywords)

#         print()
#         print_cleanup_recommendation(assets_sorted)
#         print()

#         for asset_index, asset in enumerate(assets_sorted, start=1):
#             print("-" * 120)
#             print(f"Asset {asset_index}")
#             print_asset_manual_review_block(asset, indent="  ")

#         print()

#     print()
#     print("KEY COLLISION GROUPS")
#     print("-" * 120)

#     for index, group in enumerate(key_collision_groups, start=1):
#         if index > max_key_collision_groups_to_print:
#             print("... more key collision groups not printed")
#             break

#         print("=" * 120)
#         print(f"Key Collision Group {index:02d}")
#         print("=" * 120)
#         print("unique_id:", group["unique_id"])
#         print("sha256 count:", len(group["sha256_to_assets"]))

#         for sha256, assets in group["sha256_to_assets"].items():
#             print("  sha256:", sha256)
#             print("  asset count:", len(assets))

#             for asset in assets:
#                 print("    uuid:", asset.get("uuid"))
#                 print("    original_filename:", asset.get("original_filename"))
#                 print("    filename:", asset.get("filename"))
#                 print("    date:", asset.get("date"))
#                 print("    date_added:", asset.get("date_added"))
#                 print("    file_size_bytes:", asset.get("file_size_bytes"))
#                 print("    albums:", get_asset_album_titles(asset))
#                 print("    folder_paths:", get_asset_folder_paths(asset))
#                 print("    keywords:", get_asset_keywords(asset))
#                 print("    path:", asset.get("path"))

#         print()

#     return {
#         "potential_groups": potential_groups,
#         "true_content_duplicate_groups": true_content_duplicate_groups,
#         "key_collision_groups": key_collision_groups,
#         "sha_error_assets": sha_error_assets,
#     }


# backup_duplicate_diagnostic = diagnose_potential_duplicate_groups_with_sha256_and_metadata(
#     inventory_backup,
#     "BACKUP potential duplicate diagnostic with metadata",
# )

# print()

# current_duplicate_diagnostic = diagnose_potential_duplicate_groups_with_sha256_and_metadata(
#     inventory_current,
#     "CURRENT potential duplicate diagnostic with metadata",
# )

In [8]:
# # ============================================================
# # Appendix B: Debug assets without unique ID
# # 
# # DEBUG: Dump assets without photo_library_asset_unique_id
# # ============================================================

# import os
# import time
# from collections import Counter

# def debug_dump_assets_without_photo_library_asset_unique_id(inventory, label, max_print=80):
#     missing_assets = [
#         asset
#         for asset in inventory["assets"]
#         if asset.get("photo_library_asset_unique_id") is None
#     ]

#     reason_counter = Counter()

#     print(label)
#     print("-" * 120)
#     print("assets without photo_library_asset_unique_id:", len(missing_assets))
#     print()

#     for asset in missing_assets:
#         path = asset.get("path")
#         original_filename = asset.get("original_filename")
#         filename = asset.get("filename")
#         date = asset.get("date")
#         file_size_bytes = asset.get("file_size_bytes")
#         adjustment_signature = asset.get("adjustment_signature")

#         if original_filename is None and filename is None:
#             reason_counter["missing filename and original_filename"] += 1

#         if date is None:
#             reason_counter["missing date"] += 1

#         if path is None:
#             reason_counter["path is None"] += 1
#         elif not os.path.exists(path):
#             reason_counter["path does not exist"] += 1

#         if file_size_bytes is None:
#             reason_counter["file_size_bytes is None"] += 1

#         if adjustment_signature is None:
#             reason_counter["adjustment_signature is None"] += 1

#     print("reason counter:")
#     for reason, count in reason_counter.most_common():
#         print(f"  {reason}: {count}")

#     print()
#     print("missing asset details:")
#     print("-" * 120)

#     for index, asset in enumerate(missing_assets[:max_print], start=1):
#         path = asset.get("path")

#         print(f"{index:02d}.")
#         print("  uuid:", asset.get("uuid"))
#         print("  original_filename:", asset.get("original_filename"))
#         print("  filename:", asset.get("filename"))
#         print("  date:", asset.get("date"))
#         print("  date_added:", asset.get("date_added"))
#         print("  path:", path)
#         print("  path_exists:", None if path is None else os.path.exists(path))
#         print("  file_size_bytes:", asset.get("file_size_bytes"))
#         print("  adjustment_signature:", asset.get("adjustment_signature"))
#         print("  is_movie:", asset.get("is_movie"))
#         print("  hasadjustments:", asset.get("hasadjustments"))
#         print("  path_edited:", asset.get("path_edited"))
#         print("  asset_scope:", asset.get("asset_scope"))
#         print("  albums:", list((asset.get("albums") or {}).values()))
#         print("  folders:", list((asset.get("folders") or {}).values()))
#         print("-" * 120)

# debug_dump_assets_without_photo_library_asset_unique_id(
#     inventory_current,
#     "CURRENT DEFAULT assets without photo_library_asset_unique_id",
# )